Datascraper for the weather data

In [1]:
import requests
import pandas as pd
from datetime import datetime

# ---------------------------------------------------------
# 1. Konfiguration
# ---------------------------------------------------------

API_BASE = "https://api.srgssr.ch/srf-meteo/v2"  # V2-Endpunkt
TOKEN = "DEIN_BEARER_TOKEN_HIER"                 # aus developer.srgssr.ch

# Koordinaten von Luzern wie in deinem Link
LAT = 47.0456
LON = 8.3087


# ---------------------------------------------------------
# 2. Hilfsfunktion: nächstes Geolocation-Id für Koordinaten
# ---------------------------------------------------------

def get_geolocation_id(lat: float, lon: float) -> str:
    """
    Holt die SRF-Meteo-Geolocation-ID für gegebene Koordinaten.
    """
    url = f"{API_BASE}/geolocations"
    headers = {"Authorization": f"Bearer {TOKEN}"}
    params = {"latitude": lat, "longitude": lon}

    r = requests.get(url, headers=headers, params=params)
    r.raise_for_status()
    data = r.json()

    if not data:
        raise ValueError("Keine Geolocations für diese Koordinaten gefunden.")

    loc = data[0]

    # je nach API-Version steckt die ID direkt oder in 'geolocation'
    if isinstance(loc, dict) and "geolocation" in loc:
        return loc["geolocation"]["id"]
    else:
        return loc["id"]


# ---------------------------------------------------------
# 3. Forecast holen und in DataFrame packen
# ---------------------------------------------------------

def get_daily_forecast_dataframe(lat: float, lon: float) -> pd.DataFrame:
    """
    Holt die Tagesvorhersage für eine Position und baut ein DataFrame mit:
    - Datum
    - Regenmenge (mm)
    - Sonnenscheindauer (h)
    - Tmin, Tmax (°C)
    """
    geolocation_id = get_geolocation_id(lat, lon)

    # V2-Forecast-Endpunkt. Falls dein Account noch V1 nutzt,
    # wäre es typischerweise: "https://api.srgssr.ch/srf-meteo/forecast/{geolocation_id}"
    url = f"{API_BASE}/forecast/{geolocation_id}"
    headers = {"Authorization": f"Bearer {TOKEN}"}

    r = requests.get(url, headers=headers)
    r.raise_for_status()
    data = r.json()

    # Tagesdaten liegen in der Regel unter forecast["days"]
    days = data["forecast"]["days"]

    rows = []
    for d in days:
        # date_time ist ein ISO-String, wir schneiden auf Datum oder parsen zu datetime
        dt_raw = d.get("date_time")
        try:
            date = datetime.fromisoformat(dt_raw).date()
        except Exception:
            # Fallback: einfach die ersten 10 Zeichen (YYYY-MM-DD)
            date = dt_raw[:10] if dt_raw else None

        rows.append({
            "date": date,
            "rain_mm": d.get("RRR_MM"),   # Niederschlagssumme
            "sunshine_h": d.get("SUN_H"), # Sonnenscheinstunden
            "temp_min_C": d.get("TN_C"),  # Tmin
            "temp_max_C": d.get("TX_C"),  # Tmax
        })

    df = pd.DataFrame(rows)
    return df


# ---------------------------------------------------------
# 4. Beispielaufruf für Luzern
# ---------------------------------------------------------

if __name__ == "__main__":
    df_luzern = get_daily_forecast_dataframe(LAT, LON)
    print(df_luzern)


HTTPError: 401 Client Error: Unauthorized for url: https://api.srgssr.ch/srf-meteo/v2/geolocations?latitude=47.0456&longitude=8.3087